# Student Performance Analytics & Early Prediction

This notebook follows the internship workflow: load documented data, inspect data quality, perform EDA, and evaluate models. It uses the UCI Mathematics subset and intentionally excludes G1/G2 from model inputs for the early-prediction use case.

Run `pip install -r ../requirements.txt` from the project root first.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.pipeline import load_data, evaluate_and_train
pd.set_option('display.max_columns', 100)

## 1. Load and validate data
We record shape, types, missing values, duplicates, and target range before modeling.

In [ ]:
data = load_data()
print('Shape:', data.shape)
display(data.head())
display(data.dtypes.value_counts().rename_axis('dtype').to_frame('columns'))
display(data.isna().sum().sort_values(ascending=False).head(15).to_frame('missing'))
print('Duplicate rows:', data.duplicated().sum())
print('G3 range:', data['G3'].min(), 'to', data['G3'].max())

## 2. Exploratory data analysis
Charts are descriptive. Associations should not be described as causal effects.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(data=data, x='G3', bins=11, kde=True, ax=ax)
ax.set_title('Final Mathematics grade distribution')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.boxplot(data=data, x='studytime', y='G3', ax=ax)
ax.set_title('Final grade by weekly study-time category')
plt.show()

In [ ]:
numeric = data.select_dtypes(include='number')
fig, ax = plt.subplots(figsize=(11, 8))
sns.heatmap(numeric.corr(), cmap='vlag', center=0, ax=ax)
ax.set_title('Numeric feature correlation')
plt.show()

## 3. Train and evaluate models
A fixed 20% test set is held out. Five-fold CV on training data selects the lowest mean MAE. G1 and G2 are excluded to avoid changing the early-prediction question. Preprocessing is fitted within each training fold via pipelines.

In [ ]:
metrics, best_name, X, y, X_test, y_test, best_model = evaluate_and_train(data)
display(metrics)
print('Selected by mean CV MAE:', best_name)

## 4. Interpretation checklist
- Compare ML models with the dummy baseline.
- Explain MAE in grade points and report RMSE/R² alongside it.
- Do not invent metrics; copy the actual run output into the report.
- Discuss dataset size, two-school context, potential bias, uncertainty, and the need for local validation.
- Never use this prototype for automated high-stakes decisions about students.